<a href="https://colab.research.google.com/github/huseyincenik/john_snow_labs/blob/main/generating_conll_files_from_pretrained_models/notebooks/prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prediction - Trained Model Inference

This notebook uses a trained custom NER model to make predictions on new texts.

**Google Drive Integration:**
- All files are saved to Google Drive
- Files are read from Google Drive
- Trained models are loaded from Google Drive
- All code is embedded in this notebook (no external Python files required)

## Steps:
1. **Google Drive Connection** - Mount Google Drive
2. **Setup & License** - Spark NLP Healthcare license and environment setup
3. **Model Loading** - Load trained custom NER model from Google Drive
4. **Prediction Pipeline** - Create prediction pipeline
5. **Make Predictions** - Run predictions on new texts
6. **Visualize Results** - Display and save prediction results

**Requirements:**
- `training.ipynb` notebook must be run first
- Trained model must exist at `models/trained/custom_ner_model` in Google Drive


## 1. Google Drive Connection


In [1]:
# Mount Google Drive
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

# Set project folder in Google Drive
PROJECT_FOLDER = '/content/drive/MyDrive/john_snow_labs_ner'
os.makedirs(PROJECT_FOLDER, exist_ok=True)

# Change working directory
os.chdir(PROJECT_FOLDER)

# Create folder structure
for folder in ['predictions']:
    os.makedirs(folder, exist_ok=True)

print(f"✅ Google Drive mounted")
print(f"✅ Project folder: {PROJECT_FOLDER}")
print(f"✅ Current directory: {os.getcwd()}")


Mounted at /content/drive
✅ Google Drive mounted
✅ Project folder: /content/drive/MyDrive/john_snow_labs_ner
✅ Current directory: /content/drive/MyDrive/john_snow_labs_ner


## 2. Setup & License Configuration


In [2]:
import json
import os

# Load license keys from Google Drive
license_path = f'{PROJECT_FOLDER}/spark_jsl.json'
if not os.path.exists(license_path):
    print("❌ License file not found!")
    print("Please upload spark_jsl.json to Google Drive at the project folder")
    print("You can upload it manually or use the following code:")
    print("from google.colab import files")
    print("uploaded = files.upload()")
    raise FileNotFoundError(f"License file not found at {license_path}")

with open(license_path) as f:
    license_keys = json.load(f)

# Set license keys as environment variables
locals().update(license_keys)
os.environ.update(license_keys)

print("✅ License keys loaded")
print(f"JSL Version: {license_keys.get('JSL_VERSION', 'N/A')}")
print(f"Public Version: {license_keys.get('PUBLIC_VERSION', 'N/A')}")


✅ License keys loaded
JSL Version: 6.1.1
Public Version: 6.1.3


In [3]:
# Install Java (required for Spark)
import subprocess

try:
    java_version = subprocess.check_output(['java', '-version'], stderr=subprocess.STDOUT, text=True)
    print(f"✅ Java is already installed: {java_version.split(chr(10))[0]}")

    if 'JAVA_HOME' not in os.environ:
        java_paths = [
            "/usr/lib/jvm/java-11-openjdk-amd64",
            "/usr/lib/jvm/java-8-openjdk-amd64",
            "/usr/lib/jvm/default-java"
        ]
        for path in java_paths:
            if os.path.exists(path):
                os.environ["JAVA_HOME"] = path
                print(f"✅ Set JAVA_HOME to: {path}")
                break
except Exception as e:
    print(f"Java check failed: {e}")
    print("Installing Java 11...")
    os.system('apt-get update -qq > /dev/null 2>&1')
    os.system('apt-get -y install -qq openjdk-11-jdk > /dev/null 2>&1')
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
    print("✅ Java 11 installation attempted")

# Check GPU availability
# gpu_check = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
# has_gpu = gpu_check.returncode == 0

# if has_gpu:
#     print("🚀 GPU detected! Installing PyTorch with CUDA support...")
#     %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
# else:
#     print("Installing PyTorch (CPU version)...")
#     %pip install -q torch torchvision torchaudio

# Install PySpark and Spark NLP
%pip install --upgrade -q pyspark==3.4.1 spark-nlp==$PUBLIC_VERSION

# Install Spark NLP Healthcare
%pip install --upgrade -q spark-nlp-jsl==$JSL_VERSION --extra-index-url https://pypi.johnsnowlabs.com/$SECRET

# Install additional dependencies
%pip install -q pandas numpy

print("✅ All libraries installed successfully!")
# if has_gpu:
#     print("✅ GPU-accelerated PyTorch installed")


Java check failed: [Errno 2] No such file or directory: 'java'
Installing Java 11...
✅ Java 11 installation attempted
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 MB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 737.0/737.0 kB 54.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 0.8.3 requires pyspark[connect]~=3.5.1, but you have pyspark 3.4.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 676.4 kB/s eta 0:00:00
✅ All libraries installed successfully!


In [4]:
import sparknlp
import sparknlp_jsl
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer, WordEmbeddingsModel
from sparknlp_jsl.annotator import MedicalNerModel, NerConverter
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.sql.types import StringType
from pyspark.sql import Row
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        gpu_name = torch.cuda.get_device_name(0)
        print(f"🚀 GPU Detected: {gpu_name}")
    else:
        print("⚠️  No GPU detected. Using CPU mode.")
except ImportError:
    print("⚠️  PyTorch not available. GPU check skipped.")
    gpu_available = False

# Spark configuration
params = {
    "spark.driver.memory": "8G",
    "spark.kryoserializer.buffer.max": "2000M",
    "spark.driver.maxResultSize": "2000M",
    "spark.sql.execution.arrow.pyspark.enabled": "true",
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer"
}

if gpu_available:
    params.update({
        "spark.jsl.settings.pretrained.cache_folder": f"{PROJECT_FOLDER}/cache_pretrained",
        "spark.jsl.settings.storage.cluster_tmp_dir": f"{PROJECT_FOLDER}/cache_pretrained",
        "spark.jsl.settings.annotator.gpu": "true"
    })
    print("✅ GPU acceleration enabled in Spark configuration")

# Start Spark session
try:
    print("Starting Spark session...")
    spark = sparknlp_jsl.start(license_keys['SECRET'], params=params)
    spark.sparkContext.setLogLevel("ERROR")

    print(f"✅ Spark NLP Version: {sparknlp.version()}")
    print(f"✅ Spark NLP JSL Version: {sparknlp_jsl.version()}")
    print("✅ Spark session initialized successfully")

except Exception as e:
    print(f"❌ Error starting Spark session: {e}")
    raise

spark


🚀 GPU Detected: Tesla T4
✅ GPU acceleration enabled in Spark configuration
Starting Spark session...
✅ Spark NLP Version: 6.1.3
✅ Spark NLP JSL Version: 6.1.1
✅ Spark session initialized successfully


In [5]:
# Load trained model from Google Drive
model_path = f"{PROJECT_FOLDER}/models/trained/custom_ner_model"

if not os.path.exists(model_path):
    print(f"❌ Model not found: {model_path}")
    print("Please run training.ipynb first to train the model.")
    raise FileNotFoundError(f"Model not found at {model_path}")

print(f"Loading trained model from {model_path}...")
custom_model = MedicalNerModel.load(model_path)
print("✅ Model loaded successfully")

# Load embeddings (required for the model)
print("Loading embeddings...")
embeddings = WordEmbeddingsModel.pretrained("embeddings_clinical", "en", "clinical/models")\
    .setInputCols(["sentence", "token"])\
    .setOutputCol("embeddings")
print("✅ Embeddings loaded")


Loading trained model from /content/drive/MyDrive/john_snow_labs_ner/models/trained/custom_ner_model...
✅ Model loaded successfully
Loading embeddings...
embeddings_clinical download started this may take some time.
Approximate size to download 1.6 GB
[OK!]
✅ Embeddings loaded


## 4. Create Prediction Pipeline


In [6]:

"""
NER Pipeline Module
Creates and executes Spark NLP Healthcare NER pipeline with multiple models
"""

import sparknlp
import sparknlp_jsl
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer, WordEmbeddingsModel
from sparknlp_jsl.annotator import (
    MedicalNerModel,
    NerConverter
)
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from typing import Optional, Dict, List
import warnings
warnings.filterwarnings('ignore')


class NERPipeline:
    """NER Pipeline with multiple pre-trained models"""

    def __init__(self, spark: SparkSession, license_secret: Optional[str] = None):
        """
        Initialize NER Pipeline

        Args:
            spark: SparkSession instance
            license_secret: Spark NLP Healthcare license secret (if not already configured)
        """
        self.spark = spark
        self.license_secret = license_secret
        self.pipeline = None
        self.models = {}

    def create_pipeline(self, prioritize_posology_deid: bool = True):
        """
        Create NER pipeline with multiple models

        Args:
            prioritize_posology_deid: If True, posology and deid models take priority
        """
        # Document Assembler
        document_assembler = DocumentAssembler()\
            .setInputCol("text")\
            .setOutputCol("document")\
            .setCleanupMode("shrink")

        # Sentence Detector
        sentence_detector = SentenceDetector()\
            .setInputCols(["document"])\
            .setOutputCol("sentence")\
            .setExplodeSentences(True)

        # Tokenizer
        tokenizer = Tokenizer()\
            .setInputCols(["sentence"])\
            .setOutputCol("token")

        # Word Embeddings (required for MedicalNerModel)
        # Clinical embeddings are used for all NER models
        print("Loading clinical word embeddings...")
        word_embeddings = WordEmbeddingsModel.pretrained("embeddings_clinical", "en", "clinical/models")\
            .setInputCols(["sentence", "token"])\
            .setOutputCol("embeddings")
        print("✅ Clinical embeddings loaded")

        # NER Models
        # Note: MedicalNerModel requires 3 inputs: document, token, and word_embeddings
        # 1. Clinical NER Model
        print("Loading ner_clinical model...")
        ner_clinical = MedicalNerModel.pretrained("ner_clinical", "en", "clinical/models")\
            .setInputCols(["document", "token", "embeddings"])\
            .setOutputCol("ner_clinical")

        # 2. DeID Generic Augmented Model
        print("Loading ner_deid_generic_augmented model...")
        ner_deid = MedicalNerModel.pretrained("ner_deid_generic_augmented", "en", "clinical/models")\
            .setInputCols(["document", "token", "embeddings"])\
            .setOutputCol("ner_deid")

        # 3. Posology Model (for Drug and Dosage)
        print("Loading ner_posology model...")
        ner_posology = MedicalNerModel.pretrained("ner_posology", "en", "clinical/models")\
            .setInputCols(["document", "token", "embeddings"])\
            .setOutputCol("ner_posology")

        # Store models
        self.models = {
            'clinical': ner_clinical,
            'deid': ner_deid,
            'posology': ner_posology
        }

        # Create pipeline stages
        # Note: word_embeddings must come before NER models
        stages = [
            document_assembler,
            sentence_detector,
            tokenizer,
            word_embeddings,  # Required for MedicalNerModel
            ner_clinical,
            ner_deid,
            ner_posology
        ]

        # If prioritizing, we need to merge results
        # For now, we'll run all models and merge in post-processing
        if prioritize_posology_deid:
            # Add NerConverter for each model
            ner_converter_clinical = NerConverter()\
                .setInputCols(["document", "token", "ner_clinical"])\
                .setOutputCol("chunk_clinical")

            ner_converter_deid = NerConverter()\
                .setInputCols(["document", "token", "ner_deid"])\
                .setOutputCol("chunk_deid")

            ner_converter_posology = NerConverter()\
                .setInputCols(["document", "token", "ner_posology"])\
                .setOutputCol("chunk_posology")

            stages.extend([
                ner_converter_clinical,
                ner_converter_deid,
                ner_converter_posology
            ])

        self.pipeline = Pipeline(stages=stages)
        return self.pipeline

    def fit_transform(self, data):
        """
        Fit and transform data through pipeline

        Args:
            data: Spark DataFrame with 'text' column

        Returns:
            Transformed DataFrame with NER results
        """
        if self.pipeline is None:
            raise ValueError("Pipeline not created. Call create_pipeline() first.")

        model = self.pipeline.fit(data)
        result = model.transform(data)
        return result

    def filter_posology_entities(self, ner_result, keep_entities: List[str] = ["Drug", "Dosage"]):
        """
        Filter posology entities to keep only Drug and Dosage

        Args:
            ner_result: NER result from posology model
            keep_entities: List of entity types to keep

        Returns:
            Filtered NER results
        """
        # This would be implemented based on the actual structure of NER results
        # For now, this is a placeholder
        return ner_result

    def merge_ner_results(self, result_df, prioritize_posology_deid: bool = True):
        """
        Merge results from multiple NER models with priority

        Priority order (if prioritize_posology_deid=True):
        1. Posology (Drug, Dosage)
        2. DeID (PHI entities)
        3. Clinical (other clinical entities)

        Args:
            result_df: DataFrame with NER results from all models
            prioritize_posology_deid: Whether to prioritize posology and deid

        Returns:
            DataFrame with merged NER results
        """
        # This is a complex operation that requires:
        # 1. Extracting entities from each model
        # 2. Resolving conflicts based on priority
        # 3. Creating a unified entity list

        # For now, return the original dataframe
        # Full implementation would merge chunks with priority logic
        return result_df

    def extract_entities(self, result_df) -> List[Dict]:
        """
        Extract entities from pipeline results

        Args:
            result_df: DataFrame with NER results

        Returns:
            List of entity dictionaries with text_id, begin, end, chunk, entity
        """
        entities = []

        # Extract entities from each model
        # This is a simplified version - actual implementation would need
        # to handle the Spark DataFrame structure properly

        return entities

## 5. Prepare Input Texts


In [44]:
import os
from pyspark.sql import Row
from pyspark.sql.functions import col, explode
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer, WordEmbeddingsModel
from sparknlp_jsl.annotator import MedicalNerModel, NerConverter
from pyspark.ml import Pipeline

# ---------------------------
# 1. Define project folder and model path
# ---------------------------
# PROJECT_FOLDER = "/kaggle/working"  # veya kendi klasörünüz
model_path = f"{PROJECT_FOLDER}/models/trained/custom_ner_model"

if not os.path.exists(model_path):
    raise FileNotFoundError(f"❌ Model not found: {model_path}. Please train it first.")
print(f"✅ Loading trained model from {model_path}...")

# Load trained custom NER model
custom_model = MedicalNerModel.load(model_path)
print("✅ Custom NER model loaded")

# Load embeddings required for the model
print("Loading embeddings...")
embeddings = WordEmbeddingsModel.pretrained("embeddings_clinical", "en", "clinical/models")\
    .setInputCols(["sentence", "token"])\
    .setOutputCol("embeddings")
print("✅ Embeddings loaded")

# ---------------------------
# 2. Sample texts
# ---------------------------
sample_texts = [ "The patient was prescribed Aspirin 100mg twice daily for pain management and was advised to monitor blood pressure due to hypertension.", "Dr. Smith recommended Metformin 500mg for type 2 diabetes mellitus and suggested regular HbA1c testing.", "The patient has a history of hypertension, hyperlipidemia, and is currently taking Lisinopril 10mg once daily, along with Atorvastatin 20mg at night.", "Patient presents with chest pain, shortness of breath, and palpitations. ECG shows ST elevation, and troponin levels are elevated.", "The medication dosage was increased to 20mg per day after consultation. The patient also started Omeprazole 40mg daily for gastroesophageal reflux disease.", "Administered Vancomycin 1g IV every 12 hours. Monitor renal function and complete blood count during therapy.", "Patient diagnosed with chronic kidney disease and is on Furosemide 40mg daily. Blood urea nitrogen and creatinine levels should be monitored.", "The patient received a flu vaccine and was advised to continue Vitamin D 2000 IU daily for bone health.", "Amoxicillin 500mg thrice daily was prescribed for bacterial pneumonia. Patient reported mild nausea as a side effect.", "The patient underwent MRI of the brain due to persistent headaches and dizziness.", "Patient presents with fever, chills, and productive cough. Chest X-ray confirms lobar pneumonia.", "The cardiologist prescribed Clopidogrel 75mg daily for post-stent thrombosis prevention.", "Patient has Type 1 Diabetes and is taking Insulin glargine 20 units at bedtime.", "The patient shows signs of anemia. Hemoglobin levels were 9.5 g/dL and iron supplements were recommended.", "Patient reports intermittent palpitations and shortness of breath. Echocardiogram shows mild mitral regurgitation.", "The patient started Hydroxychloroquine 200mg daily for rheumatoid arthritis treatment.", "Patient has a history of asthma and uses Albuterol inhaler 90 mcg as needed.", "The patient received a COVID-19 booster and is advised to continue monitoring oxygen saturation.", "Patient diagnosed with hypothyroidism and is taking Levothyroxine 50 mcg every morning.", "The patient presents with abdominal pain and diarrhea. Stool culture tested positive for Salmonella.", "Administered Ceftriaxone 2g IV once daily for severe bacterial infection.", "Patient has a history of hyperthyroidism and is on Methimazole 10mg daily.", "The patient reports insomnia and anxiety. Prescribed Diazepam 5mg at night.", "Patient was treated with Prednisone 20mg daily for acute exacerbation of COPD.", "Patient undergoing chemotherapy with Cisplatin 70mg/m2 every 3 weeks. Monitor renal function and complete blood count.", "The patient presents with rash, itching, and swelling. Prescribed Cetirizine 10mg daily." ]

text_df = spark.createDataFrame([Row(text=text) for text in sample_texts])
print(f"✅ Created DataFrame with {text_df.count()} texts")




✅ Loading trained model from /content/drive/MyDrive/john_snow_labs_ner/models/trained/custom_ner_model...
✅ Custom NER model loaded
Loading embeddings...
embeddings_clinical download started this may take some time.
Approximate size to download 1.6 GB
[OK!]
✅ Embeddings loaded
✅ Created DataFrame with 26 texts


## 6. Run Predictions


In [45]:

# ---------------------------
# 3. Pipeline components
# ---------------------------
document_assembler = DocumentAssembler()\
    .setInputCol("text")\
    .setOutputCol("document")\
    .setCleanupMode("shrink")

sentence_detector = SentenceDetector()\
    .setInputCols(["document"])\
    .setOutputCol("sentence")\
    .setExplodeSentences(True)

tokenizer = Tokenizer()\
    .setInputCols(["sentence"])\
    .setOutputCol("token")

# NerConverter to get chunks
ner_converter = NerConverter()\
    .setInputCols(["document", "token", "ner_custom"])\
    .setOutputCol("chunks")

# Set input/output columns for the custom model
custom_model.setInputCols(["document", "token", "embeddings"]).setOutputCol("ner_custom")

# ---------------------------
# 4. Create pipeline
# ---------------------------
pipeline = Pipeline(stages=[
    document_assembler,
    sentence_detector,
    tokenizer,
    embeddings,
    custom_model,
    ner_converter
])

# ---------------------------
# 5. Fit and transform
# ---------------------------
model_pipeline = pipeline.fit(text_df)
predictions = model_pipeline.transform(text_df)
print("✅ Predictions completed")

# ---------------------------
# 6. Show results
# ---------------------------
predictions.select(
    "text",
    "chunks.result"
).show(truncate=200)

# Optional: explode entities for easier inspection
exploded = predictions.select(
    col("text"),
    explode(col("chunks.result")).alias("entity")
)
exploded.show(truncate=200)


✅ Predictions completed
+-----------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+
|                                                                                                                                                       text|         result|
+-----------------------------------------------------------------------------------------------------------------------------------------------------------+---------------+
|                    The patient was prescribed Aspirin 100mg twice daily for pain management and was advised to monitor blood pressure due to hypertension.|             []|
|                                                    Dr. Smith recommended Metformin 500mg for type 2 diabetes mellitus and suggested regular HbA1c testing.|             []|
|      The patient has a history of hypertension, hyperlipidemia, and is currently taking Lisinopril 10mg 

## 7. NER Visualization with spark-nlp-display

We'll use **spark-nlp-display** to visualize NER results interactively.

### Key Features:
- Visual representation of named entities
- Color-coded entity labels
- HTML export for sharing results
- Customizable label colors and filters

In [46]:
# Install spark-nlp-display if not already installed
# !pip install spark-nlp-display

from sparknlp_display import NerVisualizer

# Initialize the visualizer
visualizer = NerVisualizer()

print("✅ NER Visualizer initialized")

✅ NER Visualizer initialized


In [71]:
# Read and parse CoNLL file from Google Drive
conll_file_path = f"{PROJECT_FOLDER}/data/conll/conll2003_text_file.conll"

def parse_conll_file(file_path, num_documents=3, max_tokens_per_doc=100):
    """
    Parse CoNLL file and extract sample texts (partial documents)

    Args:
        file_path: Path to CoNLL file
        num_documents: Number of documents to extract
        max_tokens_per_doc: Maximum number of tokens per document (default: 100)
                           This limits the text size to avoid processing huge documents

    CoNLL format:
    - Each line: TOKEN POS_TAG CHUNK_TAG NER_TAG
    - Empty line separates sentences
    - -DOCSTART- marks document boundaries
    - B- prefix: Beginning of entity
    - I- prefix: Inside/continuation of entity
    - O: Outside any entity
    """
    documents = []
    current_doc_tokens = []
    current_doc_entities = []

    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            # Document boundary
            if line.startswith('-DOCSTART-'):
                if current_doc_tokens:
                    # Save previous document (limited to max_tokens_per_doc)
                    text = ' '.join(current_doc_tokens[:max_tokens_per_doc])
                    documents.append({
                        'text': text,
                        'tokens': current_doc_tokens[:max_tokens_per_doc],
                        'entities': current_doc_entities[:max_tokens_per_doc]
                    })
                    current_doc_tokens = []
                    current_doc_entities = []

                if len(documents) >= num_documents:
                    break
                continue

            # Empty line (sentence boundary)
            if not line:
                continue

            # Parse token line
            parts = line.split()
            if len(parts) >= 4:
                token = parts[0]
                ner_tag = parts[3]

                current_doc_tokens.append(token)
                current_doc_entities.append(ner_tag)

    # Add last document if exists (limited to max_tokens_per_doc)
    if current_doc_tokens and len(documents) < num_documents:
        text = ' '.join(current_doc_tokens[:max_tokens_per_doc])
        documents.append({
            'text': text,
            'tokens': current_doc_tokens[:max_tokens_per_doc],
            'entities': current_doc_entities[:max_tokens_per_doc]
        })

    return documents

# Load sample documents from CoNLL file
# Process only first 100 tokens from each document for faster processing
if os.path.exists(conll_file_path):
    sample_docs = parse_conll_file(conll_file_path, num_documents=3, max_tokens_per_doc=100)
    print(f"✅ Loaded {len(sample_docs)} sample documents from CoNLL file")
    print(f"   (Limited to first 100 tokens per document for faster processing)")

    # Display first document structure
    if sample_docs:
        print("\n📝 First document example:")
        print(f"Text length: {len(sample_docs[0]['text'])} characters")
        print(f"Number of tokens: {len(sample_docs[0]['tokens'])}")
        print(f"\nFirst 10 tokens and their NER tags:")
        for i in range(min(10, len(sample_docs[0]['tokens']))):
            token = sample_docs[0]['tokens'][i]
            entity = sample_docs[0]['entities'][i]
            prefix = "  "
            if entity.startswith('B-'):
                prefix = "▶️"  # Beginning of entity
            elif entity.startswith('I-'):
                prefix = "  ▪️"  # Inside entity
            print(f"{prefix} {token:20s} -> {entity}")
else:
    print(f"⚠️ CoNLL file not found at {conll_file_path}")
    sample_docs = []

✅ Loaded 1 sample documents from CoNLL file
   (Limited to first 100 tokens per document for faster processing)

📝 First document example:
Text length: 552 characters
Number of tokens: 100

First 10 tokens and their NER tags:
   PROCEDURES           -> O
   PERFORMED            -> O
   :                    -> O
▶️ Colonosco            -> B-TEST
   py                   -> O
   .                    -> O
   INDICATIONS          -> O
   :                    -> O
▶️ Renewed              -> B-PROBLEM
  ▪️ sympto               -> I-PROBLEM


### 7.2 Run Predictions on CoNLL Examples with LightPipeline

We'll use LightPipeline for faster inference on individual texts.

In [72]:
# Create LightPipeline for faster predictions
from sparknlp.base import LightPipeline
import pandas as pd

# Create LightPipeline from the fitted model
light_pipeline = LightPipeline(model_pipeline)
print("✅ LightPipeline created")

# Run predictions on CoNLL samples
conll_predictions = []

if sample_docs:
    for idx, doc in enumerate(sample_docs):
        print(f"\n{'='*60}")
        print(f"Processing document {idx + 1}/{len(sample_docs)}")
        print(f"{'='*60}")

        # Get prediction using fullAnnotate for complete output
        result = light_pipeline.fullAnnotate(doc['text'])

        if result and len(result) > 0:
            conll_predictions.append(result[0])

            # Extract chunks and entities
            chunks = []
            entities = []
            begin_pos = []
            end_pos = []
            confidence_scores = []

            for chunk in result[0].get('chunks', []):
                chunks.append(chunk.result)
                entities.append(chunk.metadata.get('entity', 'N/A'))
                begin_pos.append(chunk.begin)
                end_pos.append(chunk.end)
                confidence_scores.append(chunk.metadata.get('confidence', 'N/A'))

            # Create DataFrame for better visualization
            df = pd.DataFrame({
                'chunk': chunks,
                'entity': entities,
                'begin': begin_pos,
                'end': end_pos,
                'confidence': confidence_scores
            })

            print(f"\n📊 Detected {len(df)} entities:")
            if not df.empty:
                display(df)
else:
    print("⚠️ No CoNLL documents to process")

print(f"\n✅ Predictions completed for {len(conll_predictions)} documents")

✅ LightPipeline created

Processing document 1/1

📊 Detected 17 entities:


,chunk,entity,begin,end,confidence
0,Colonosco,TEST,23,31,0.9982
1,Renewed sympto,PROBLEM,52,65,0.96005
2,Inflammatory Bowel Disea,PROBLEM,109,132,0.88240004
3,conventional thera,TREATMENT,157,174,0.8192
4,sulfasalazi,TREATMENT,189,199,0.997
5,cortiso,TREATMENT,206,212,0.9902
6,local thera,TREATMENT,219,229,0.93495
7,the procedu,TREATMENT,287,297,0.93435
8,bleedi,PROBLEM,381,386,0.9915
9,infecti,PROBLEM,393,399,0.9976



✅ Predictions completed for 1 documents


### 7.3 Visualize NER Results with spark-nlp-display

Now let's visualize the predictions with entity highlighting:
- Different colors for different entity types
- Clear boundary markers showing where entities begin and end
- Understanding of B- (Begin) and I- (Inside) prefixes in the CoNLL format

In [73]:
# Visualize predictions for each CoNLL document
if conll_predictions:
    for idx, prediction in enumerate(conll_predictions):
        print(f"\n{'='*80}")
        print(f"📊 VISUALIZATION FOR DOCUMENT {idx + 1}")
        print(f"{'='*80}")

        # Display using NerVisualizer
        # label_col='chunks' refers to the NER chunks output
        # document_col='document' refers to the document column
        visualizer.display(
            prediction,
            label_col='chunks',
            document_col='document',
            return_html=False
        )

        print("\n" + "-"*80)
else:
    print("⚠️ No predictions to visualize")

print("\n✅ Visualization complete")


📊 VISUALIZATION FOR DOCUMENT 1



--------------------------------------------------------------------------------

✅ Visualization complete


### 7.4 Understanding CoNLL Format: B- and I- Prefixes

Let's analyze the entity prefixes to understand how entities are tagged in CoNLL format:

**Prefix Explanation:**
- **B-ENTITY**: Marks the **Beginning** of an entity (e.g., B-TREATMENT, B-PROBLEM)
- **I-ENTITY**: Marks tokens **Inside** or continuing an entity (e.g., I-TREATMENT, I-PROBLEM)
- **O**: Token is **Outside** any entity (regular text)

**Example:**
```
Token          NER Tag
---------      -----------
conventional   B-TREATMENT  ← Beginning of treatment entity
thera          I-TREATMENT  ← Inside/continuation of treatment entity
py             O            ← Outside (end of entity)
```

This format helps the model understand:
1. Where entities start (B- prefix)
2. Which tokens belong together as one entity (I- prefix)
3. Multi-token entities (consecutive B- and I- tags)

In [74]:
# Analyze entity prefixes from CoNLL file
if sample_docs:
    print("🔍 Analyzing Entity Prefixes in CoNLL Format\n")

    # Collect entity type statistics
    entity_types = {}
    b_prefix_count = 0
    i_prefix_count = 0
    o_count = 0

    for doc in sample_docs:
        for tag in doc['entities']:
            if tag == 'O':
                o_count += 1
            elif tag.startswith('B-'):
                b_prefix_count += 1
                entity_type = tag[2:]  # Remove 'B-' prefix
                entity_types[entity_type] = entity_types.get(entity_type, 0) + 1
            elif tag.startswith('I-'):
                i_prefix_count += 1

    print("📊 Tag Distribution:")
    print(f"   B- tags (Beginning): {b_prefix_count}")
    print(f"   I- tags (Inside):    {i_prefix_count}")
    print(f"   O tags (Outside):    {o_count}")
    print(f"   Total tokens:        {b_prefix_count + i_prefix_count + o_count}")

    print("\n📋 Entity Types Found:")
    for entity_type, count in sorted(entity_types.items(), key=lambda x: x[1], reverse=True):
        print(f"   {entity_type:15s}: {count} occurrences")

    # Show example of multi-token entity
    print("\n💡 Example of Multi-token Entity from CoNLL:")
    if sample_docs:
        doc = sample_docs[0]
        in_entity = False
        entity_tokens = []
        entity_type = ""

        for i, (token, tag) in enumerate(zip(doc['tokens'], doc['entities'])):
            if tag.startswith('B-'):
                if entity_tokens:  # Print previous entity
                    print(f"   Entity: '{' '.join(entity_tokens)}' → Type: {entity_type}")
                    entity_tokens = []

                entity_tokens = [token]
                entity_type = tag[2:]
                in_entity = True

            elif tag.startswith('I-') and in_entity:
                entity_tokens.append(token)

            elif in_entity:
                # Entity ended
                print(f"   Entity: '{' '.join(entity_tokens)}' → Type: {entity_type}")
                entity_tokens = []
                in_entity = False

            # Show first 3 entities
            if entity_type and len(entity_tokens) == 0 and not in_entity:
                if i > 50:  # Stop after finding a few examples
                    break
else:
    print("⚠️ No CoNLL documents loaded")

🔍 Analyzing Entity Prefixes in CoNLL Format

📊 Tag Distribution:
   B- tags (Beginning): 17
   I- tags (Inside):    12
   O tags (Outside):    71
   Total tokens:        100

📋 Entity Types Found:
   PROBLEM        : 7 occurrences
   TREATMENT      : 6 occurrences
   TEST           : 4 occurrences

💡 Example of Multi-token Entity from CoNLL:
   Entity: 'Colonosco' → Type: TEST
   Entity: 'Renewed sympto' → Type: PROBLEM
   Entity: 'Inflammatory Bowel Disea' → Type: PROBLEM
   Entity: 'conventional thera' → Type: TREATMENT
   Entity: 'sulfasalazi' → Type: TREATMENT
   Entity: 'cortiso' → Type: TREATMENT
   Entity: 'local thera' → Type: TREATMENT
   Entity: 'the procedu' → Type: TREATMENT


In [75]:
# Extract NER results using PySpark arrays_zip for structured output
# This method provides detailed information about each entity:
# - sentence_id: Which sentence the entity belongs to
# - chunk: The actual text of the entity
# - begin: Start position in the text
# - end: End position in the text
# - ner_label: The entity type/label

from pyspark.sql import functions as F

# First, let's run predictions on CoNLL samples using PySpark DataFrame
# Create a DataFrame from sample documents
if sample_docs:
    print("🔄 Processing CoNLL samples with PySpark DataFrame...\n")

    # Create DataFrame from sample documents
    conll_texts = [{"text": doc['text']} for doc in sample_docs]
    conll_df = spark.createDataFrame(conll_texts)

    # Transform using the pipeline
    conll_predictions_df = model_pipeline.transform(conll_df)

    # Extract structured NER results using arrays_zip
    print("📊 Extracting Structured NER Results:\n")
    print("="*80)

    structured_results = conll_predictions_df.select(
        F.explode(
            F.arrays_zip(
                conll_predictions_df.chunks.result,
                conll_predictions_df.chunks.begin,
                conll_predictions_df.chunks.end,
                conll_predictions_df.chunks.metadata
            )
        ).alias("cols")
    ).select(
        F.expr("cols['3']['sentence']").alias("sentence_id"),
        F.expr("cols['0']").alias("chunk"),
        F.expr("cols['1']").alias("begin"),
        F.expr("cols['2']").alias("end"),
        F.expr("cols['3']['entity']").alias("ner_label")
    ).filter("ner_label != 'O'")  # Filter out non-entities

    # Display results
    print("\n📋 All Detected Entities:")
    structured_results.show(50, truncate=False)

    # Show entity statistics
    print("\n📈 Entity Distribution:")
    print("-"*80)
    structured_results.groupBy("ner_label")\
        .count()\
        .orderBy(F.desc("count"))\
        .show(truncate=False)

    # Show entities by sentence
    print("\n📝 Entities Grouped by Sentence:")
    print("-"*80)
    structured_results.groupBy("sentence_id")\
        .agg(
            F.count("*").alias("entity_count"),
            F.collect_list("ner_label").alias("entity_types")
        )\
        .orderBy("sentence_id")\
        .show(truncate=False)

    print("\n✅ Structured extraction complete!")

else:
    print("⚠️ No CoNLL documents available for structured extraction")

🔄 Processing CoNLL samples with PySpark DataFrame...

📊 Extracting Structured NER Results:


📋 All Detected Entities:
+-----------+------------------------+-----+---+---------+
|sentence_id|chunk                   |begin|end|ner_label|
+-----------+------------------------+-----+---+---------+
|0          |Colonosco               |23   |31 |TEST     |
|0          |Renewed sympto          |52   |65 |PROBLEM  |
|0          |Inflammatory Bowel Disea|109  |132|PROBLEM  |
|0          |conventional thera      |157  |174|TREATMENT|
|0          |sulfasalazi             |189  |199|TREATMENT|
|0          |cortiso                 |206  |212|TREATMENT|
|0          |local thera             |219  |229|TREATMENT|
|0          |the procedu             |287  |297|TREATMENT|
|0          |bleedi                  |381  |386|PROBLEM  |
|0          |infecti                 |393  |399|PROBLEM  |
|0          |bowel perforati         |406  |420|PROBLEM  |
|0          |aspiration pneumon      |427  |444|PROBLEM 